In [1]:
import urllib.request
import pandas as pd
import subprocess
import numpy as np
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import csc_matrix
import sys

data_root="/home/mcn26/palmer_scratch/tabula_data"
data_out="/gpfs/gibbs/pi/reilly/tabula_data"

# PreProcess

In [2]:
dat=pd.read_csv(f"{data_root}/seelig_mpra_unpro.tsv",index_col=0,sep="\t")

In [3]:
dat

,Gene Name,Gene,A9_A2_A2,A6_A2_A2,A2_B1_A2,A5_B2_A2,A1_B2_A2,A7_B3_A2,A4_A1_A2,A5_A1_A2,...,A3_F2_F8,A5_F4_F8,A9_F4_F8,A6_F5_F8,A4_F6_F8,A7_F6_F8,A12_F7_F8,A5_F7_F8,A5_F8_F8,A6_F8_F8
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1340,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1341,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1342,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1343,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
dat.index=dat['Gene']
dat.drop(['Gene Name','Gene'],axis=1,inplace=True)

In [5]:
dat=dat.stack().reset_index()

In [6]:
dat.rename({'level_1':'cell_bc',0:'umis_mpra_bc'},axis=1,inplace=True)

In [7]:
dat['umis']=dat['umis'].astype(int)

In [8]:
dat

,Gene,cell_bc,umis_mpra_bc
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0.0
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0.0
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0.0
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0.0
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0.0
...,...,...,...
14310795,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0.0
14310796,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0.0
14310797,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0.0
14310798,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0.0


In [8]:
dat['umis_mpra_bc']=dat['umis_mpra_bc'].astype(int)

In [9]:
dat

,Gene,cell_bc,umis_mpra_bc
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0
...,...,...,...
14310795,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0
14310796,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0
14310797,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0
14310798,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0


In [9]:
acceptible_cells_and_cell_type=pd.read_csv("acceptible_cells_and_cell_type.tsv",sep="\t")

In [10]:
acceptible_cells_and_cell_type

,barcode,cell_type
0,A9_A2_A2,HEPG2
1,A6_A2_A2,K562
2,A2_B1_A2,HEPG2
3,A5_B2_A2,K562
4,A1_B2_A2,HEPG2
...,...,...
9722,A7_F6_F8,K562
9723,A12_F7_F8,HEPG2
9724,A5_F7_F8,K562
9725,A5_F8_F8,K562


In [11]:
acceptible_cells_and_cell_type.rename({'barcode':'cell_bc'},axis=1,inplace=True)

In [12]:
#make sure no dup cell barcodes : would cause duplicated data
assert len(acceptible_cells_and_cell_type["cell_bc"])==len(acceptible_cells_and_cell_type["cell_bc"].unique())

In [13]:
merged=dat.merge(acceptible_cells_and_cell_type,how="inner",on="cell_bc")

In [14]:
print(f'retained={len(merged)/len(dat)*100}%.')

retained=91.41917293233082%.


In [15]:
merged

,Gene,cell_bc,umis_mpra_bc,cell_type
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2
...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562


In [16]:
merged.rename({'Gene':'cre_id'},axis=1,inplace=True)

In [17]:
merged['rep_id'] = 'rep1'
merged

,cre_id,cell_bc,umis_mpra_bc,cell_type,rep_id
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2,rep1
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562,rep1
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2,rep1
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562,rep1
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2,rep1
...,...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562,rep1
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2,rep1
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562,rep1
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562,rep1


In [18]:
merged=merged[['cell_bc','rep_id','cre_id','cell_type','umis']].copy()

Now merge in DNA

In [23]:
dna=pd.read_csv("dna.tsv",sep="\t",index_col=0)

In [24]:
plus_DNA=pd.merge(merged,dna,on="cre_id",how="left",validate="many_to_one")

In [25]:
plus_DNA

,cell_bc,rep_id,cre_id,cell_type,umis,reads_DNA
0,A9_A2_A2,NaN,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
1,A6_A2_A2,NaN,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
2,A2_B1_A2,NaN,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
3,A5_B2_A2,NaN,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
4,A1_B2_A2,NaN,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
...,...,...,...,...,...,...
13082810,A7_F6_F8,NaN,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
13082811,A12_F7_F8,NaN,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451
13082812,A5_F7_F8,NaN,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
13082813,A5_F8_F8,NaN,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451


In [28]:
assert not any(plus_DNA.loc[:, plus_DNA.columns != 'rep_id'].isnull().any(axis=1))

In [32]:
plus_DNA.to_csv(f'/gpfs/gibbs/pi/reilly/tabula_data/seelig/seelig_with_dna.tsv',sep='\t')
#plus_DNA.to_csv(f'~/seelig_with_dna.tsv',sep='\t')

# Set up for models

In [20]:
DATA_PATH = '/gpfs/gibbs/pi/reilly/tabula_data'
PARAMS_PATH = '../../model_fitting'

class DataSet(dict):
    def __init__(self, path):
        self.filepath = path
        self.parquet = pq.ParquetFile(self.filepath)
    
    def __getitem__(self, key):
        try:
            return self.parquet.read([key]).to_pandas()[key]
        except:
            raise KeyError

    def __reduce__(self):
        #return self.parquet.read().to_pandas().__reduce__()
        return (self.__class__, (self.filepath, ))

In [21]:
merged

,cre_id,cell_bc,umis_mpra_bc,cell_type,rep_id
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2,rep1
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562,rep1
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2,rep1
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562,rep1
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2,rep1
...,...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562,rep1
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2,rep1
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562,rep1
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562,rep1


## Combo

In [23]:
 counts_groupby_cre = merged

 counts_groupby_cre.to_csv('%s/seelig/seelig_counts_grouped.txt' % DATA_PATH, sep='\t', index=False)

counts_groupby_cre = pd.read_table('%s/seelig/seelig_counts_grouped.txt' % DATA_PATH)
counts_groupby_cre 

,cre_id,cell_bc,umis_mpra_bc,cell_type,rep_id
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2,rep1
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562,rep1
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2,rep1
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562,rep1
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2,rep1
...,...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562,rep1
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2,rep1
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562,rep1
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562,rep1


In [24]:
table = pa.Table.from_pandas(counts_groupby_cre)
pq.write_table(table, '%s/seelig/seelig_mpra_counts_grouped.parq' % DATA_PATH)
counts_parq_grouped = DataSet('%s/seelig/seelig_mpra_counts_grouped.parq' % DATA_PATH)

In [25]:
sys.getsizeof(counts_parq_grouped)

80

In [27]:
count = 1
with open('%s/seelig_combo_model_params.txt' % PARAMS_PATH, 'w') as file:
    for model_type in ['poisson','zi_poisson','negative_binomial','zi_negative_binomial']:
        line = (str(count) + '\t' + model_type  + '\t' +  'umis_mpra_bc~C(cre_id)+C(cell_type)+0' + '\t' + 'rep_id' + '\t' + str(50000) + '\t' + 'COMBO'+ '\t' + 'ALL_seelig' + '\t' + 'seelig/seelig_mpra_counts_grouped.parq')
        file.write(line + "\n")
        print(line)
        count += 1
        line = (str(count) + '\t' + model_type  + '\t' +  'umis_mpra_bc~C(cre_id)*C(cell_type)+0' + '\t' + 'rep_id' + '\t' + str(50000) + '\t' + 'COMBO'+ '\t' + 'ALL_seelig' + '\t' + 'seelig/seelig_mpra_counts_grouped.parq')
        file.write(line + "\n")
        print(line)
        count += 1

1	poisson	umis_mpra_bc~C(cre_id)+C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
2	poisson	umis_mpra_bc~C(cre_id)*C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
3	zi_poisson	umis_mpra_bc~C(cre_id)+C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
4	zi_poisson	umis_mpra_bc~C(cre_id)*C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
5	negative_binomial	umis_mpra_bc~C(cre_id)+C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
6	negative_binomial	umis_mpra_bc~C(cre_id)*C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
7	zi_negative_binomial	umis_mpra_bc~C(cre_id)+C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq
8	zi_negative_binomial	umis_mpra_bc~C(cre_id)*C(cell_type)+0	rep_id	50000	COMBO	ALL_seelig	seelig/seelig_mpra_counts_grouped.parq


## Cell type individual groupings

In [28]:
counts_groupby_cell_agg = counts_groupby_cre.groupby(['cell_type']).agg(
    Sum=('umis_mpra_bc', 'sum'), Size=('umis_mpra_bc','size'), Mean=('umis_mpra_bc', 'mean')
)
counts_groupby_cell_agg

,Sum,Size,Mean
cell_type,,,
HEPG2,635287,6104955,0.104061
K562,326539,6977860,0.046796


In [29]:
cell_types = list(counts_groupby_cell_agg.index)
len(cell_types)

2

In [30]:
for i in cell_types:
    print(i)
    counts = counts_groupby_cre[counts_groupby_cre.cell_type == i]
    table = pa.Table.from_pandas(counts)
    pq.write_table(table, '%s/seelig/cell_type/seelig_mpra_counts_%s.parq' % (DATA_PATH, i))

HEPG2
K562


In [31]:
count = 1
with open('%s/seelig_cell_type_individual_models_params.txt' % PARAMS_PATH, 'w') as file:
    for i in cell_types:
        for model_type in ['poisson','zi_poisson','negative_binomial','zi_negative_binomial']:
            line = (str(count) + '\t' + model_type  + '\t' +  'umis_mpra_bc~C(cre_id)+0' + '\t' + 'rep_id' + '\t' + str(50000) + '\t' + 'Seelig_CELLTYPE'+ '\t' + i + '\t' + 'seelig/cell_type/seelig_mpra_counts_%s.parq' % i)
            file.write(line + "\n")
            print(line)
            count += 1

1	poisson	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	HEPG2	seelig/cell_type/seelig_mpra_counts_HEPG2.parq
2	zi_poisson	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	HEPG2	seelig/cell_type/seelig_mpra_counts_HEPG2.parq
3	negative_binomial	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	HEPG2	seelig/cell_type/seelig_mpra_counts_HEPG2.parq
4	zi_negative_binomial	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	HEPG2	seelig/cell_type/seelig_mpra_counts_HEPG2.parq
5	poisson	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	K562	seelig/cell_type/seelig_mpra_counts_K562.parq
6	zi_poisson	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	K562	seelig/cell_type/seelig_mpra_counts_K562.parq
7	negative_binomial	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	K562	seelig/cell_type/seelig_mpra_counts_K562.parq
8	zi_negative_binomial	umis_mpra_bc~C(cre_id)+0	rep_id	50000	Seelig_CELLTYPE	K562	seelig/cell_type/seelig_mpra_counts_K562.parq


In [32]:
sys.getsizeof(counts_groupby_cre)

4790716659

In [33]:
import sys

## CRE individual groupings

In [34]:
counts_groupby_cre_agg = counts_groupby_cre.groupby(['cre_id']).agg(
    Sum=('umis_mpra_bc', 'sum'), Size=('umis_mpra_bc','size'), Mean=('umis_mpra_bc', 'mean')
)
counts_groupby_cre_agg

,Sum,Size,Mean
cre_id,,,
AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC,48,9727,0.004935
AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGCTGAACTTTGGCTGTTGGTATGGTGACGTGACATAGCTTTGCAACGTACTGTCTGTAACCTTGGACTTTGCACAAACGTACAAAGCATGCCGGAGGGGAA,3556,9727,0.365580
AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTCTGGAGGCTGAGGCAGGAAAATGGCGGGAACCCGAGAGGCGGAGCTTGCAGTGAGCCGATATCGCGCCACTGAACTCCAGCCCGGACAACAGAGCGAGAC,116,9727,0.011926
AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAGGCAGGCCGGAATGGTCAACCTTTGGTCTTTGTACCGTCATGTTGACCTCGTCTGGACGGTTGAACTTTGCCCGTGTGCATTGGTACACTCGGTATGTAC,172,9727,0.017683
AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGCGCGATCAAACAAAGGTTGAGCGAAAATCCACCCGTGCAAACATGTCCGGGCATGCCTGCTGGCGAACGTGCGAACTCCGACCGGAGAGTTTGGCGCACG,162,9727,0.016655
...,...,...,...
TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACTTCTGATGAGTAATCACGCAATCGGTTCCGCGAAGTAAGATCTGCTATCGGACGCGGCGCTTCCTTCTTATATTGGGAAATCGCTTCATTAGTATGAGGC,147,9727,0.015113
TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGCGCGCCTGAAGGCGATTATGGCGCGCGCGCCTGACACGGGGTATTGTCCTGGAGTTATGCGGCTACAGGAATGGGACGGCGCCACTGCGGGGTTTGCCGG,11,9727,0.001131
TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGGGCACGTGACCTCTGTCCGGTAATGTTTGAACAAAGGTCACATGCCTGGGCATGTCCTTTGAACAAAGCGAACATTACATAAACGTTCACAGGTCACCTG,637,9727,0.065488


In [35]:
cres = list(counts_groupby_cre_agg.index)
len(cres)

1345

In [36]:
for i in cres:
    print(i)
    counts = counts_groupby_cre[counts_groupby_cre.cre_id == i]
    table = pa.Table.from_pandas(counts)
    pq.write_table(table, '%s/seelig/cre/seelig_mpra_counts_%s.parq' % (DATA_PATH, i))

...


In [37]:
count = 1
with open('%s/seelig_cre_individual_models_params.txt' % PARAMS_PATH, 'w') as file:
    for i in cres:
        for model_type in ['poisson','zi_poisson','negative_binomial','zi_negative_binomial']:
            line = (str(count) + '\t' + model_type  + '\t' +  'umis_mpra_bc~C(cell_type)+0' + '\t' + 'rep_id' + '\t' + str(50000) + '\t' + 'CRE' + '\t' + i + '\t' + 'seelig/cre/seelig_mpra_counts_%s.parq' % i)
            file.write(line + "\n")
            print(line)
            count += 1

1	poisson	umis_mpra_bc~C(cell_type)+0	rep_id	50000	CRE	AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC	seelig/cre/seelig_mpra_counts_AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC.parq
2	zi_poisson	umis_mpra_bc~C(cell_type)+0	rep_id	50000	CRE	AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC	seelig/cre/seelig_mpra_counts_AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC.parq
3	negative_binomial	umis_mpra_bc~C(cell_type)+0	rep_id	50000	CRE	AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC	seelig/cre/seelig_mpra